# 07 Test: the single evaluation on held-out searches

**Purpose.** Evaluate the final models once on the 20,000 test searches, which no earlier notebook has evaluated. This is the only notebook that reads test labels.

**Procedure (fixed before the run).**
1. **Validation check first.** The models of this pipeline run must reproduce the validation numbers we report (`validation_clean`, NDCG@10), within fixed tolerances: 0.00001 for untrained rankings, 0.001 for LightGBM models (deterministic, so normally exact) and 0.002 for the neural models (GPU training is not bit-identical). If any model is outside its tolerance, the notebook stops **before reading any test label**.
2. **Test scores** were written by notebooks 05 (LightGBM) and 05b (neural) without labels; here they are joined with the test labels and evaluated with the same evaluator as validation.
3. **Rankings:** random order, lagged history popularity, the LambdaMART base, the untrained rule "own products first, then the base", M1 (base + pooled history), M1 + CF (the final LambdaMART), the feature MLP and the regularised history attention; supporting: M3 (history by type and window), M1 + all six other feature groups and the deployable full model.
4. **Populations:** `test_clean` (an engaged candidate, no clamped final gap) is the main population; all test searches and test searches with a final gap over 30 s are sensitivity checks.
5. **Paired comparisons** (per-search differences with 95% bootstrap intervals): M1 - base, M1 - rule, M1 + CF - M1, feature MLP - M1 + CF, attention - feature MLP, base - lagged popularity; supporting: M3 - M1, M1 + all six - M1, full model - M1.
6. Every result is reported; nothing is changed or selected on the test result.

**Runtime:** CPU, about 5 minutes.

In [ ]:
# Setup: on Colab, clone the repository (shared code in jdsearch.py) and use Google Drive as the working folder.
import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/repo").is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/liamkunert/BT4222-group17-jdsearch-submission.git", "/content/repo"], check=True)
    sys.path.insert(0, "/content/repo")
else:
    sys.path.insert(0, str(Path.cwd().resolve().parent))
import jdsearch as jd

paths = jd.setup()
jd.require(paths, "results.csv", "processed/search_table.parquet", "processed/split_v1.csv",
           "processed/features_v3_test.parquet", "processed/test/lambdamart_test_scores.parquet",
           "processed/test/neural_test_scores.parquet")

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.options.display.float_format = "{:,.4f}".format

## 1. Validation check (before any test label is read)

Reference values: the `validation_clean` NDCG@10 of each model as reported (our development run). This run's values come from `results.csv`, written by notebooks 04, 05 and 05b.

In [ ]:
# name in results.csv: (reported validation_clean NDCG@10, tolerance)
REFERENCE = {
    "random order": (0.08418, 1e-5),
    "history popularity, lag 1 day": (0.33745, 1e-5),
    "base, all training": (0.34995, 0.001),
    "rule: own products first, then the trained base": (0.38400, 0.001),
    "M1, all training": (0.38868, 0.001),
    "M3, all training": (0.39002, 0.001),
    "M1 + CF, all training": (0.39176, 0.001),
    "deployable full, all training": (0.39054, 0.001),
    "M1 + all six groups, all training": (0.39069, 0.001),
    "feature MLP": (0.38893, 0.002),
    "history attention v3 (regularised)": (0.3867, 0.002),
}
check = pd.DataFrame({name: {"reported": value, "this run": jd.logged(paths, name), "tolerance": tolerance}
                      for name, (value, tolerance) in REFERENCE.items()}).T
check["difference"] = check["this run"] - check["reported"]
check["within tolerance"] = check.difference.abs() <= check.tolerance + 1e-9
display(check)
assert check["within tolerance"].all(), "validation numbers not reproduced: stop before reading test labels"
print("validation check passed")

## 2. Test scores and labels

The score files keep the row order of the v3 test feature file; the labels are read from that file here.

In [ ]:
searches = pd.read_parquet(paths.processed / "search_table.parquet")
assignment = pd.read_csv(paths.processed / "split_v1.csv", index_col="session_id")
test_ids = assignment.index[assignment.split == "test"]
test_sizes = searches.loc[test_ids, "n_candidates"]
frame = pq.read_table(paths.processed / "features_v3_test.parquet",
                      columns=["session_id", "wid", "label", "pop_history_lag1d"]).to_pandas()
trees = pd.read_parquet(paths.processed / "test/lambdamart_test_scores.parquet")
neural = pd.read_parquet(paths.processed / "test/neural_test_scores.parquet")
for scores in (trees, neural):
    assert (scores.session_id.to_numpy() == frame.session_id.to_numpy()).all()
    assert (scores.wid.to_numpy() == frame.wid.to_numpy()).all()
base = trees["base"].to_numpy()
RANKINGS = {
    "random order": np.zeros(len(frame)),
    "lagged history popularity": frame.pop_history_lag1d.fillna(0).to_numpy(dtype=float),
    "LambdaMART base": base,
    "rule: own products first, then the base": (trees.pooled_product.fillna(0).to_numpy(dtype=float)
                                                + 0.999 * (base - base.min()) / (base.max() - base.min())),
    "M1: base + pooled history": trees["M1"].to_numpy(),
    "M1 + CF (final LambdaMART)": trees["M1 + CF"].to_numpy(),
    "feature MLP": neural["mlp"].to_numpy(),
    "history attention v3 (regularised)": neural["attention_v3"].to_numpy(),
    "M3: history by type and window": trees["M3"].to_numpy(),
    "M1 + all six groups": trees["M1 + all six"].to_numpy(),
    "deployable full model": trees["deployable full"].to_numpy(),
}
per_search = {name: jd.evaluate(frame.assign(score=score), "score", expected_sizes=test_sizes)
              for name, score in RANKINGS.items()}
all_ids = per_search["random order"].index
clean_ids = all_ids[~searches.final_gap_clamped.reindex(all_ids).to_numpy()]
long_gap_ids = clean_ids[(searches.final_gap_s.reindex(clean_ids) > 30).to_numpy()]
print(f"test searches with an engaged candidate: {len(all_ids):,}; test_clean {len(clean_ids):,}; "
      f"final gap over 30 s {len(long_gap_ids):,}")

## 3. Test results

NDCG@10 with 95% bootstrap intervals, Recall@10 and MRR on `test_clean`, next to the reported validation value; NDCG@10 on all test searches as sensitivity.

In [ ]:
VALIDATION_NAME = {"random order": "random order", "lagged history popularity": "history popularity, lag 1 day",
                   "LambdaMART base": "base, all training",
                   "rule: own products first, then the base": "rule: own products first, then the trained base",
                   "M1: base + pooled history": "M1, all training", "M1 + CF (final LambdaMART)": "M1 + CF, all training",
                   "feature MLP": "feature MLP", "history attention v3 (regularised)": "history attention v3 (regularised)",
                   "M3: history by type and window": "M3, all training",
                   "M1 + all six groups": "M1 + all six groups, all training",
                   "deployable full model": "deployable full, all training"}
rows = {}
for name, per in per_search.items():
    row = jd.summarise(per.loc[clean_ids]).iloc[0].to_dict()
    row["validation_clean"] = jd.logged(paths, VALIDATION_NAME[name])
    row["test, all searches"] = per.ndcg.mean()
    rows[name] = row
    for split_name, ids in [("test_clean", clean_ids), ("test", all_ids)]:
        jd.log_result(paths, "07", "test", name, split_name, per.loc[ids], notes="single test evaluation")
results = pd.DataFrame(rows).T
results["searches"] = results.searches.astype(int)
results

## 4. Paired comparisons on test

In [ ]:
PAIRS = [("M1: base + pooled history", "LambdaMART base"),
         ("M1: base + pooled history", "rule: own products first, then the base"),
         ("M1 + CF (final LambdaMART)", "M1: base + pooled history"),
         ("feature MLP", "M1 + CF (final LambdaMART)"),
         ("history attention v3 (regularised)", "feature MLP"),
         ("LambdaMART base", "lagged history popularity"),
         ("M3: history by type and window", "M1: base + pooled history"),
         ("M1 + all six groups", "M1: base + pooled history"),
         ("deployable full model", "M1: base + pooled history")]
rows = {}
for a, b in PAIRS:
    for population, ids in [("test_clean", clean_ids), ("final gap over 30 s", long_gap_ids)]:
        if population != "test_clean" and (a, b) != PAIRS[0]:
            continue  # the final-gap sensitivity only for the personal gain
        d = jd.paired_difference(per_search[a].loc[ids], per_search[b].loc[ids])
        rows[(f"{a}  minus  {b}", population)] = {"difference": d[0], "low": d[1], "high": d[2]}
comparisons = pd.DataFrame(rows).T
comparisons.to_csv(paths.outputs / "test_comparisons.csv")
results.to_csv(paths.outputs / "test_results.csv")
comparisons

## Summary

To be written after the single test run.